## Inspect the upstream checkpoint and measure baseline answers

In [ ]:
from pathlib import Path
import torch,json,random,math,time,sys,os
from transformers import AutoTokenizer,AutoModelForCausalLM
from huggingface_hub import HfApi,hf_hub_download
base1="LiquidAI/LFM2.5-1.2B-Instruct"; api1=HfApi(); rev1=api1.model_info(base1).sha
work1=Path("/kaggle/working/Pratham-1B");work1.mkdir(exist_ok=True)
dev1=torch.device("cuda:1" if torch.cuda.device_count()>1 else "cuda:0")
t1=AutoTokenizer.from_pretrained(base1,revision=rev1)
b1=AutoModelForCausalLM.from_pretrained(base1,revision=rev1,torch_dtype=torch.float16,attn_implementation="eager").to(dev1)
params1=b1.num_parameters(); print("BASE_READY",params1,flush=True)
"""Reproducible short-answer regression checks for the Pratham chat release.

Pass a callable that accepts a list of role/content messages and returns only
the new assistant reply. Keep these prompts out of instruction-tuning data.
"""
import re
import unicodedata

CASES = [
    ("extract-blue", "Mira put the key in a blue box. What color is the box? Answer in one word.", {"blue"}),
    ("extract-red", "Mira put the key in a red box. What color is the box? Answer in one word.", {"red"}),
    ("extract-owner", "Isha gave her umbrella to Dev. Who received the umbrella? Answer with a name.", {"dev"}),
    ("addition", "What is two plus two? Answer with just the result.", {"4", "four"}),
    ("subtraction", "I had seven pears and gave two away. How many remain? Answer with a number.", {"5", "five"}),
    ("negation", "The vase fell but did not break. Did it break? Answer yes or no.", {"no"}),
    ("missing", "The parcel arrived yesterday. What color is it? If unspecified, answer unknown.", {"unknown"}),
]

def normalize(text):
    return re.sub(r"[^\w]+", " ", unicodedata.normalize("NFKC", text).casefold()).strip()

def evaluate(reply):
    results = []
    for name, prompt, expected in CASES:
        answer = reply([{"role": "user", "content": prompt}])
        results.append(dict(case=name, prompt=prompt, answer=answer,
                            passed=normalize(answer) in expected))
    history = [{"role": "user", "content": "My bag is green."},
               {"role": "assistant", "content": "Understood."},
               {"role": "user", "content": "Correction: it is purple now."},
               {"role": "assistant", "content": "Understood."},
               {"role": "user", "content": "What color is my bag now? Answer in one word."}]
    answer = reply(history)
    results.append(dict(case="multi-turn-correction", prompt=history, answer=answer,
                        passed=normalize(answer) == "purple"))
    return results

@torch.inference_mode()
def reply1(messages):
    x=t1.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_tensors="pt")
    if not torch.is_tensor(x): x=x["input_ids"]
    x=x.to(dev1)
    y=b1.generate(input_ids=x,attention_mask=torch.ones_like(x),max_new_tokens=24,do_sample=False,pad_token_id=t1.eos_token_id)
    return t1.decode(y[0,x.shape[1]:],skip_special_tokens=True).strip()
base_eval=evaluate(reply1); print("BASE_EVAL",base_eval,flush=True)


## Train the response-token correction and select on held-out loss

In [ ]:
import hashlib, numpy as np
from torch.nn import functional as F
from torch import nn
random.seed(29);torch.manual_seed(29)
data_id="databricks/databricks-dolly-15k";data_rev=api1.dataset_info(data_id).sha
dpath=hf_hub_download(data_id,"databricks-dolly-15k.jsonl",repo_type="dataset",revision=data_rev)
rows1=[json.loads(x) for x in open(dpath)];random.Random(29).shuffle(rows1)
def pack1(row):
    q=row["instruction"]+("\n"+row["context"] if row["context"] else "")
    p=t1.apply_chat_template([{"role":"user","content":q}],tokenize=True,add_generation_prompt=True)
    if isinstance(p,dict) or hasattr(p,"keys"): p=p["input_ids"]
    if len(p)>300:return None
    a=t1.encode(row["response"],add_special_tokens=False)+[t1.eos_token_id]
    a=a[:384-len(p)]
    if not a:return None
    return (p+a,[-100]*len(p)+a)
examples=[v for r in rows1 if (v:=pack1(r)) is not None]
val1=examples[:128];train1=examples[128:8320]
anchors=t1.encode(" yes no is the can one answer why where what he she they both first last true false",add_special_tokens=False)
anchors=list(dict.fromkeys(anchors))
head1=b1.get_output_embeddings();vocab1=head1.weight.shape[0]
adapt=nn.Linear(len(anchors),vocab1,bias=False,dtype=torch.float32,device=dev1)
nn.init.zeros_(adapt.weight); opt1=torch.optim.AdamW(adapt.parameters(),lr=2e-5)
def loss1(batch):
    width=max(len(x) for x,_ in batch)
    ids=torch.tensor([x+[t1.eos_token_id]*(width-len(x)) for x,_ in batch],device=dev1)
    labels=torch.tensor([y+[-100]*(width-len(y)) for _,y in batch],device=dev1)[:,1:]
    with torch.no_grad():
        h=b1.model(input_ids=ids,use_cache=False).last_hidden_state[:,:-1][labels!=-100]
        z=head1(h).float()
    return F.cross_entropy(z+adapt(z[:,anchors]),labels[labels!=-100]), int((labels!=-100).sum())
def measure1():
    with torch.no_grad():
        vals=[loss1(val1[i:i+4]) for i in range(0,len(val1),4)]
    return sum(v.item()*n for v,n in vals)/sum(n for _,n in vals)
b1.eval(); before1=measure1();total1=0
best1=before1; beststep=0; trail=[]
for i in range(0,len(train1),8):
    opt1.zero_grad(set_to_none=True);v,n=loss1(train1[i:i+8]);v.backward()
    torch.nn.utils.clip_grad_norm_(adapt.parameters(),1.);opt1.step();total1+=n
    if (i+8)%1024==0 or i+8>=len(train1):
        current=measure1();trail.append([min(i+8,len(train1)),current,total1])
        if current<best1:
            best1=current;beststep=min(i+8,len(train1));torch.save(adapt.state_dict(),work1/"best.pt")
        print("TRAIN1",trail[-1],flush=True)
if beststep==0: raise ValueError("Adapter failed held-out validation; do not release")
adapt.load_state_dict(torch.load(work1/"best.pt",weights_only=True))
print("END1",len(train1),total1,before1,best1,beststep,flush=True)


## Fold the correction into the output matrix, verify quality and publish

In [ ]:
import shutil
from kaggle_secrets import UserSecretsClient
api_write=HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
assert api_write.whoami()["name"]=="oncody"
repo1="oncody/Pratham-1B-Instruct"; sp1="oncody/Pratham-1B-Chat"
release1=work1/"release";release1.mkdir(exist_ok=True)
web1=work1/"space";web1.mkdir(exist_ok=True)
onnx1="LiquidAI/LFM2.5-1.2B-Instruct-ONNX";onrev=api1.model_info(onnx1).sha
(web1/"release.json").write_text(json.dumps(dict(onnx=onnx1,onnx_revision=onrev,anchors=anchors)))
adapt.weight.detach().float().cpu().numpy().astype("<f4").tofile(web1/"adapter.f32")
# Fold the learned correction into a separate output matrix. Preserve input embeddings.
with torch.no_grad():
    W=head1.weight.detach()
    merged=nn.Linear(W.shape[1],W.shape[0],bias=False,device=dev1,dtype=W.dtype)
    basis=W[anchors].float()
    for i in range(0,len(W),2048):
        merged.weight[i:i+2048].copy_(W[i:i+2048].float()+adapt.weight[i:i+2048]@basis)
    h=torch.randn(3,W.shape[1],device=dev1,dtype=W.dtype)
    raw=head1(h).float(); expected=raw+adapt(raw[:,anchors])
    similarity=F.cosine_similarity(expected,merged(h).float()).min().item()
    assert similarity>.999
b1.config.tie_word_embeddings=False;b1.set_output_embeddings(merged);b1.eval()
after_eval=evaluate(reply1)
base_score=sum(r['passed'] for r in base_eval); after_score=sum(r['passed'] for r in after_eval)
print("QA1",base_score,after_score,after_eval,flush=True)
assert after_score>=base_score and after_eval[0]['passed'] and after_eval[1]['passed'] and after_eval[3]['passed'] and after_eval[-1]['passed'], "QA gate failed"
b1.save_pretrained(release1,safe_serialization=True);t1.save_pretrained(release1)
# Reload on the CPU and compare next-token scores.
recheck=AutoModelForCausalLM.from_pretrained(release1,torch_dtype=torch.float16).eval()
probe=torch.tensor([[20,30,40]])
with torch.no_grad(): agree=F.cosine_similarity(recheck(probe).logits[0,-1].float(),b1(probe.to(dev1)).logits[0,-1].cpu().float(),dim=0).item()
assert agree>.999
del recheck
metrics=dict(base=base1,base_revision=rev1,base_parameters=params1,export_parameters=b1.num_parameters(),trainable=adapt.weight.numel(),dataset=data_id,dataset_revision=data_rev,available_rows=len(rows1),training_examples=len(train1),validation_examples=len(val1),training_response_tokens=total1,selected_examples=beststep,selected_response_tokens=next(r[2] for r in trail if r[0]==beststep),validation_before=before1,validation_after=best1,validation_history=trail,base_answers=base_eval,pratham_answers=after_eval,merge_cosine=similarity,reload_cosine=agree)
(release1/"evaluation.json").write_text(json.dumps(metrics,indent=2))
for filename in api1.list_repo_files(base1,revision=rev1):
    if filename.upper().startswith(("LICENSE","NOTICE")) and "/" not in filename:
        path=hf_hub_download(base1,filename,revision=rev1);shutil.copy2(path,release1/filename);shutil.copy2(path,web1/filename)
card=f"""---
language: [en]
base_model: {base1}
datasets: [{data_id}]
license: other
license_name: lfm1.0
license_link: LICENSE
pipeline_tag: text-generation
library_name: transformers
---
# Pratham-1B-Instruct

Rahul Chaube adapted Liquid AI's LFM2.5-1.2B-Instruct for English conversation. [Live chat](https://huggingface.co/spaces/{sp1}) · [Training and evaluation code](https://github.com/Rahulchaube1/-Pratham-117M-/tree/main/pratham_1b).

## How it was trained

The original {params1:,}-parameter pretrained model was frozen. A linear correction learned from selected vocabulary logits adjusted the predicted next-token distribution. Only {adapt.weight.numel():,} coefficients were optimized. The exported checkpoint folds this correction into a separate output embedding matrix for ordinary Transformers inference. Modified model weights and configuration are distributed with Liquid AI's original license. 
   
The training run used {len(train1):,} shuffled English examples from Databricks Dolly 15k, seed 29, lengths up to 384, and {total1:,} response tokens. Prompt tokens were excluded from the loss. A separate 128-example validation split selected the checkpoint at {beststep:,} examples. See evaluation.json for pinned input revisions, selected token count, validation history and unedited answers. Dolly was created by Databricks and shared under [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).

## Measured quality

Held-out token cross-entropy: {before1:.4f} before adaptation, {best1:.4f} after. The upstream model passed {base_score}/8 exact-format checks and the adapted model passed {after_score}/8. These eight tests cover extraction, arithmetic, negation, missing information and dialogue corrections. A correct explanation can fail an exact-format check. This small test cannot establish general factual accuracy. The browser combines an upstream quantized ONNX checkpoint with the trained correction; its outputs can differ from full precision.

## Run locally

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
repo = "{repo1}"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo, device_map="auto", dtype="auto")
inputs = tok.apply_chat_template([{{"role":"user","content":"What is two plus two?"}}], add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
out = model.generate(**inputs, do_sample=False, max_new_tokens=64)
print(tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))
```
"""
(release1/"README.md").write_text(card)
(web1/"index.html").write_text("<!doctype html><html lang=\"en\"><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\"><title>Pratham-1B Chat</title><style>body{max-width:720px;margin:7vh auto;padding:0 20px;font:17px/1.5 system-ui;background:#faf9f5;color:#172622}h1{font-size:2.3em}small{color:#495e54}textarea{width:100%;box-sizing:border-box;padding:14px;font:inherit}button{background:#17573e;color:white;border:0;padding:10px 22px;border-radius:6px}button:disabled{opacity:.5}p{white-space:pre-wrap}#status{margin:18px 0}</style><body ><h1>Pratham-1B Chat</h1><small>A chat model adapted by Rahul Chaube from Liquid AI’s LFM2.5-1.2B-Instruct. Replies are generated on your device.</small><div id=\"status\" role=\"status\">Starting…</div><div id=\"log\" aria-live=\"polite\"></div><form><label>Your message<textarea required rows=\"2\"></textarea></label><button disabled>Send</button></form><script type=\"module\" src=\"./app.js\"></script></body></html>")
(web1/"app.js").write_text("import {AutoModelForCausalLM,AutoTokenizer,LogitsProcessor,LogitsProcessorList,TextStreamer,env} from 'https://cdn.jsdelivr.net/npm/@huggingface/transformers@4.0.1/dist/transformers.min.js';\nenv.allowLocalModels=false;\nconst status=document.querySelector('#status'),form=document.querySelector('form'),send=form.querySelector('button'),box=form.querySelector('textarea'),log=document.querySelector('#log');\nconst cfg=await (await fetch('./release.json')).json();\nlet mdl,tok,processors,pending;const history=[];\nclass PrathamCorrection extends LogitsProcessor {\n constructor(weights){super();this.weights=weights}\n _call(ids,scores){const z=scores.data;const x=cfg.anchors.map(i=>z[i]);if(this.weights.length!==z.length*x.length)throw Error('Adapter vocabulary mismatch');\n for(let i=0;i<z.length;i++){let delta=0;for(let j=0;j<x.length;j++)delta+=this.weights[i*x.length+j]*x[j];z[i]+=delta}return scores}\n}\nasync function load(){\n const progress=e=>{if(e.status==='progress'&&e.total)status.textContent='Loading '+e.file+' · '+Math.round(e.loaded/e.total*100)+'%'};\n const id=cfg.onnx,opts={revision:cfg.onnx_revision,progress_callback:progress};\n tok=await AutoTokenizer.from_pretrained(id,opts);\n const useGPU=!!await navigator.gpu?.requestAdapter();\n try{mdl=await AutoModelForCausalLM.from_pretrained(id,{...opts,device:useGPU?'webgpu':'wasm',dtype:'q4'})}\n catch(err){if(!useGPU)throw err;mdl=await AutoModelForCausalLM.from_pretrained(id,{...opts,device:'wasm',dtype:'q4'})}\n const raw=await fetch('./adapter.f32');if(!raw.ok)throw Error('Adapter download failed');\n processors=new LogitsProcessorList();processors.push(new PrathamCorrection(new Float32Array(await raw.arrayBuffer())))}\nfunction row(who,message){const p=document.createElement('p');const tag=document.createElement('strong');tag.textContent=who+' ';const t=document.createElement('span');t.textContent=message;p.append(tag,t);log.append(p);return t}\nform.onsubmit=async e=>{e.preventDefault();let q=box.value.trim();if(!q||send.disabled)return;send.disabled=true;row('You',q);box.value='';const answer=row('Pratham','');\ntry{if(!mdl||!processors){status.textContent='Loading the model on this device…';await load()}\n const messages=[...history.slice(-8),{role:'user',content:q}];const input=tok.apply_chat_template(messages,{add_generation_prompt:true,return_dict:true});\n if(input.input_ids.dims[1]>2048)throw Error('This conversation exceeds 2,048 input tokens; shorten the message or reload for a new chat.');\n status.textContent='Generating…';let text='';const streamer=new TextStreamer(tok,{skip_prompt:true,skip_special_tokens:true,callback_function:s=>{text+=s;answer.textContent=text}});\n const ids=await mdl.generate({...input,logits_processor:processors,streamer,do_sample:false,max_new_tokens:128});\n const reply=tok.decode(ids[0].tolist().slice(input.input_ids.dims[1]),{skip_special_tokens:true}).trim();answer.textContent=reply||'[End of reply]';history.push({role:'user',content:q},{role:'assistant',content:reply});status.textContent='Ready'\n }catch(err){answer.textContent='';status.textContent='Could not generate: '+err.message}finally{send.disabled=false}};\nstatus.textContent='First message downloads the model. A desktop browser with WebGPU is recommended.';send.disabled=false;\n")
(web1/"README.md").write_text(f"---\ntitle: Pratham-1B Chat\nsdk: static\napp_file: index.html\n---\n# Pratham-1B Chat\nBrowser inference using Liquid AI's quantized LFM2.5-1.2B-Instruct and Rahul Chaube's trained Pratham score correction. [Model and evaluation](https://huggingface.co/{repo1}).\n")
api_write.create_repo(repo1,private=False,exist_ok=True);rc1=api_write.upload_folder(repo_id=repo1,folder_path=release1,commit_message="Release tested Pratham-1B-Instruct weights and measured evaluation")
api_write.create_repo(sp1,repo_type="space",space_sdk="static",private=False,exist_ok=True);api_write.upload_folder(repo_id=sp1,repo_type="space",folder_path=web1,commit_message="Serve Pratham-1B on-device chat")
print("PUBLISHED1",rc1.oid,metrics['export_parameters'],base_score,after_score,flush=True)
